# 01. LangChain tool call


> 랭체인에서도 tool 사용 가능

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("하이~")])

AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 10, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a5a1892700', 'id': 'chatcmpl-DuuO5hpFaa7kPZTKZ6rQMcdf3xHFp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02a0-6293-7a83-926c-f6c360ab46fc-0', usage_metadata={'input_tokens': 10, 'output_tokens': 11, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool

In [3]:
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    ### 단순한 주석이 아니라 랭체인에 이 함수의 기능과 입력값, 사용 방법을 알려주는 문서화 문자열
    """ 현재 시각을 반환하는 함수 

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [4]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

In [ ]:
# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools) #llm_with_tools는 도구를 사용하여 llm 답변을 생성할 수 있는 모델

In [6]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("서울은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

In [7]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_FiyYUkxuHZjeC4G84gdb8nAN', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5eaa148bd2', 'id': 'chatcmpl-DuuTRLAAgH7zHXnaIDzrq25EY3JQv', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02a5-70c4-7820-bab2-721664ede8d2-0', tool_cal

In [8]:
response.tool_calls

[{'name': 'get_current_time',
  'args': {'timezone': 'Asia/Seoul', 'location': '서울'},
  'id': 'call_FiyYUkxuHZjeC4G84gdb8nAN',
  'type': 'tool_call'}]

In [9]:
response.tool_calls[0]

{'name': 'get_current_time',
 'args': {'timezone': 'Asia/Seoul', 'location': '서울'},
 'id': 'call_FiyYUkxuHZjeC4G84gdb8nAN',
 'type': 'tool_call'}

In [10]:
tool_call = response.tool_calls[0]
tool_call["name"]

'get_current_time'

In [11]:
tool_dict[tool_call["name"]]

StructuredTool(name='get_current_time', description="현재 시각을 반환하는 함수 \n\n   Args:\n       timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함\n       location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨", args_schema=<class 'langchain_core.utils.pydantic.get_current_time'>, func=<function get_current_time at 0x106fc59d0>)

In [12]:
tool_dict[tool_call["name"]].invoke(tool_call)

Asia/Seoul (서울) 현재시각 2026-06-26 15:41:36 


ToolMessage(content='Asia/Seoul (서울) 현재시각 2026-06-26 15:41:36 ', name='get_current_time', tool_call_id='call_FiyYUkxuHZjeC4G84gdb8nAN')

In [13]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '서울'}
Asia/Seoul (서울) 현재시각 2026-06-26 15:43:05 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_FiyYUkxuHZjeC4G84gdb8nAN', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5eaa148bd2', 'id': 'chatcmpl-DuuTRLAAgH7zHXnaIDzrq25EY3JQv', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02a5-70c4-7820-bab2-721664ede8d2-0', tool_cal

In [14]:
llm_with_tools.invoke(messages)

AIMessage(content='서울의 현재 시각은 2026년 6월 26일 15시 43분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 189, 'total_tokens': 214, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5eaa148bd2', 'id': 'chatcmpl-Duua3hzs8PSXcFNa1KdoDzzW5vEf8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019f02ab-b3df-70b1-b40a-1ec6d89e03b9-0', usage_metadata={'input_tokens': 189, 'output_tokens': 25, 'total_tokens': 214, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [15]:
messages.append(llm_with_tools.invoke(messages))

In [16]:
messages

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_FiyYUkxuHZjeC4G84gdb8nAN', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5eaa148bd2', 'id': 'chatcmpl-DuuTRLAAgH7zHXnaIDzrq25EY3JQv', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02a5-70c4-7820-bab2-721664ede8d2-0', tool_cal

### 주가 예제

In [17]:
import yfinance as yf

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """ 
    주식 종목의 가격 데이터를 조회하는 함수

    Args:
        ticker (str): 주식 종목 코드 (예: AAPL)
        period (str): 주식 데이터 조회 기간 (예: 1d, 1mo, 1y)
    
    """
    stock = yf.Ticker(ticker)
    history = stock.history(period=period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [18]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
messages.append(response)

In [ ]:
messages #마지막에 tool_calls까지 추가되어 있음. tool_calls에 대한 처리를 해야함

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='서울은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_FiyYUkxuHZjeC4G84gdb8nAN', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"서울"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 134, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_5eaa148bd2', 'id': 'chatcmpl-DuuTRLAAgH7zHXnaIDzrq25EY3JQv', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--019f02a5-70c4-7820-bab2-721664ede8d2-0', tool_cal

In [26]:
response.tool_calls[0]

{'name': 'get_yf_stock_history',
 'args': {'ticker': 'TSLA', 'period': '1mo'},
 'id': 'call_LWuqoLRF3H2hNq2Ci39QeaPq',
 'type': 'tool_call'}

In [28]:
tool_call

{'name': 'get_yf_stock_history',
 'args': {'ticker': 'TSLA', 'period': '1mo'},
 'id': 'call_LWuqoLRF3H2hNq2Ci39QeaPq',
 'type': 'tool_call'}

In [29]:
!pip install tabulate

In [31]:
for tool_call in response.tool_calls:
    out1 = tool_dict[tool_call["name"]].invoke(tool_call) 
    messages.append(out1)

llm_with_tools.invoke(messages)

BadRequestError: Error code: 400 - {'error': {'message': "Invalid parameter: messages with role 'tool' must be a response to a preceeding message with 'tool_calls'.", 'type': 'invalid_request_error', 'param': 'messages.[8].role', 'code': None}}

## 실습 1. pdf_summary.py

pdf_summary 하는 함수 tool로 등록하여 사용해보기

+ summarize_text 도 langchain 사용하는 것으로 변경

In [32]:
from langchain_core.messages import HumanMessage
import os 

pdf_path = os.path.join(os.getcwd(),"samples/videoanomaly.pdf")

messages = [
    HumanMessage(f"이 {pdf_path} 문서의 저자는 누구야?"),
]


In [33]:
messages

[HumanMessage(content='이 /Users/user/Desktop/cursor 연습/5일차/samples/videoanomaly.pdf 문서의 저자는 누구야?', additional_kwargs={}, response_metadata={})]

In [ ]:
import pdf_summary as pdf_summary

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """ 
    주식 종목의 가격 데이터를 조회하는 함수

    Args:
        ticker (str): 주식 종목 코드 (예: AAPL)
        period (str): 주식 데이터 조회 기간 (예: 1d, 1mo, 1y)
    
    """
    stock = yf.Ticker(ticker)
    history = stock.history(period=period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)